# Mathematical Statistics (Wackerly, 7th Edition)
## Section 3.3 — The Expected Value of a Random Variable (or a Function of a Random Variable)

**Textbook:** Mathematical Statistics with Applications, 7th Edition
**Authors:** Dennis Wackerly, William Mendenhall, Richard Scheaffer

---

### What this section is about

Section 3.2 taught us how to build a **probability distribution** $p(y)$ — the full list of "what values can $Y$ take, and how likely is each." That is complete information, but it is a *table*. Often we want a **single number** that summarizes the distribution:

- Where is the distribution *centered*?  → the **expected value** $E[Y]$ (the mean $\mu$).
- How *spread out* is it?  → the **variance** $V(Y)$ (and its square root, the standard deviation $\sigma$).

### Roadmap

1. **Definition of $E[Y]$** — the probability-weighted average, and why it is the long-run average.
2. **Theorem 3.2** — expectation of a *function* $g(Y)$, without finding the distribution of $g(Y)$ first.
3. **Theorems 3.3–3.5** — the three rules that make expectation *linear*.
4. **Variance** — definition, intuition, and the computational shortcut (**Theorem 3.6**).
5. **Worked exercises** with simulation checks, including a callback to the *matching problem* from Section 3.2.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb, factorial
from itertools import permutations

rng = np.random.default_rng(20260826)


def expectation(y_vals, p_vals, g=lambda y: y):
    """E[g(Y)] = sum_y g(y) p(y)."""
    return sum(g(y) * p for y, p in zip(y_vals, p_vals))


def summarize(y_vals, p_vals, name='Y'):
    """Print the distribution, its mean, E[Y^2], and variance."""
    assert np.isclose(sum(p_vals), 1.0), f'probabilities sum to {sum(p_vals)}'
    mu  = expectation(y_vals, p_vals)
    EY2 = expectation(y_vals, p_vals, g=lambda y: y**2)
    var = EY2 - mu**2
    print(f'Distribution of {name}:')
    for y, p in zip(y_vals, p_vals):
        print(f'  P({name} = {y}) = {p:.4f}')
    print(f'  sum = {sum(p_vals):.4f}')
    print(f'\n  E[{name}]   = mu      = {mu:.4f}')
    print(f'  E[{name}^2]         = {EY2:.4f}')
    print(f'  V({name})   = sigma^2 = {var:.4f}')
    print(f'  sd({name})  = sigma   = {np.sqrt(var):.4f}')
    return mu, var


def plot_pmf_mean(y_vals, p_vals, title, xlabel='y'):
    """Bar chart of a pmf with the mean drawn as a balance point."""
    mu = expectation(y_vals, p_vals)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(y_vals))
    ax.bar(x, p_vals, 0.6, color='#1f4fa8', alpha=0.85)
    for xi, (yv, pv) in enumerate(zip(y_vals, p_vals)):
        ax.annotate(f'{pv:.3f}', (xi, pv), ha='center', va='bottom', fontsize=10)
    # Draw the mean as a fulcrum on the value-axis (interpolated to bar coords)
    mu_x = np.interp(mu, y_vals, x)
    ax.axvline(mu_x, color='#a8541f', linestyle='--', linewidth=2,
               label=f'mean $\\mu = {mu:.3f}$')
    ax.plot(mu_x, 0, marker='^', markersize=16, color='#a8541f', clip_on=False)
    ax.set_xticks(x)
    ax.set_xticklabels([str(v) for v in y_vals])
    ax.set_xlabel(xlabel)
    ax.set_ylabel('$P(Y = y)$')
    ax.set_title(title)
    ax.set_ylim(0, max(p_vals) * 1.2)
    ax.grid(axis='y', alpha=0.3)
    ax.legend()
    plt.show()

print('helpers ready')


---

## 1. The expected value $E[Y]$

### Definition 3.4

Let $Y$ be a discrete random variable with probability function $p(y)$. The **expected value** of $Y$ is

$$E[Y] \;=\; \sum_{y} y\, p(y),$$

provided the sum converges absolutely. We write $\mu = E[Y]$ and call it the **mean** of $Y$.

### The one-sentence idea

$E[Y]$ is the **probability-weighted average** of the values $Y$ can take. Values that are more likely pull the average toward themselves.

### Two ways to *feel* what the mean is

**(a) Long-run average.** Imagine playing the same random experiment over and over. Record $Y$ each time and keep a running average. The Law of Large Numbers says that running average settles down to $E[Y]$. So $E[Y]$ is "what you get per play, on average, in the long run" — even if $Y$ never actually equals $E[Y]$ on any single play.

**(b) Center of mass / balance point.** Put a mass of size $p(y)$ at each location $y$ on a number line. The point where the whole thing balances on a fulcrum is exactly $\mu = E[Y]$. That is why the plots below draw the mean as a triangle underneath the bars.

> **Caution.** The mean need not be a value $Y$ can take. A fair die has $E[Y] = 3.5$, which is not a face. The mean is a *summary*, not a *prediction* of any single outcome.


In [ ]:
# Fair die: mean = 3.5 is not a face, yet the running average converges to it.
y_vals = [1, 2, 3, 4, 5, 6]
p_vals = [1/6] * 6
mu, var = summarize(y_vals, p_vals, name='Y (fair die)')

N = 100_000
rolls = rng.integers(1, 7, size=N)
running_avg = np.cumsum(rolls) / np.arange(1, N + 1)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(np.arange(1, N + 1), running_avg, color='#1f4fa8', linewidth=1)
ax.axhline(mu, color='#a8541f', linestyle='--', linewidth=2, label=f'$E[Y] = {mu:.3f}$')
ax.set_xscale('log')
ax.set_xlabel('number of rolls (log scale)')
ax.set_ylabel('running average of $Y$')
ax.set_title('Law of Large Numbers: the running mean converges to $E[Y]$')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

plot_pmf_mean(y_vals, p_vals, 'Fair die — mean as the balance point', xlabel='y = face value')


---

## 2. Expectation of a function $g(Y)$ — Theorem 3.2

Often we don't care about $Y$ itself but about some **function** of it: your *winnings* $g(Y)$, the *area* $g(Y) = Y^2$, a *cost* $g(Y)$, etc.

### Theorem 3.2

Let $Y$ be discrete with probability function $p(y)$, and let $g(Y)$ be a real-valued function of $Y$. Then

$$E\bigl[g(Y)\bigr] \;=\; \sum_{y} g(y)\, p(y).$$

### Why this is a genuinely useful shortcut

The naive route would be: (1) find the distribution of the *new* variable $U = g(Y)$, then (2) compute $\sum_u u\, p_U(u)$. Theorem 3.2 says **skip step (1)** — just weight $g(y)$ by the *original* $p(y)$. This is sometimes called the **"law of the unconscious statistician"**: you can push $g$ through the expectation using the distribution you already have.

The reason it works: every outcome that produces the value $y$ contributes probability $p(y)$, and on that outcome $g(Y)$ equals $g(y)$. Grouping the sum over outcomes by the value of $y$ gives exactly $\sum_y g(y) p(y)$.

### Example

For the fair die, the "squared face value" has expectation

$$E[Y^2] = \sum_{y=1}^{6} y^2 \cdot \tfrac{1}{6} = \tfrac{1}{6}(1+4+9+16+25+36) = \tfrac{91}{6} \approx 15.17,$$

and note $E[Y^2] \ne (E[Y])^2 = 3.5^2 = 12.25$. **Expectation does not pass through nonlinear functions** — a fact we will use immediately for variance.


In [ ]:
# Theorem 3.2 in action: E[g(Y)] two ways, plus the E[Y^2] != (E[Y])^2 warning.
y_vals = [1, 2, 3, 4, 5, 6]
p_vals = [1/6] * 6

# Route 1 (Theorem 3.2): weight g(y) by the original p(y)
EY2_direct = expectation(y_vals, p_vals, g=lambda y: y**2)

# Route 2 (the long way): build the distribution of U = Y^2, then average
u_vals = [y**2 for y in y_vals]           # all distinct here
EY2_long = sum(u * (1/6) for u in u_vals)

print(f'E[Y^2] via Theorem 3.2 : {EY2_direct:.4f}')
print(f'E[Y^2] via new dist.   : {EY2_long:.4f}')
print(f'(E[Y])^2               : {expectation(y_vals, p_vals)**2:.4f}   <- different!')

# Simulation check
rolls = rng.integers(1, 7, size=500_000)
print(f'\nSimulated E[Y^2] approx {np.mean(rolls**2):.4f}')


---

## 3. The three linearity rules — Theorems 3.3, 3.4, 3.5

These are the workhorses. Let $c$ be a constant and $g, g_1, \dots, g_k$ be functions of $Y$.

| # | Rule | Statement | Plain English |
|---|---|---|---|
| **3.3** | Constant | $E[c] = c$ | The expected value of a constant is that constant. |
| **3.4** | Scaling | $E[c\,g(Y)] = c\,E[g(Y)]$ | Constants factor out of an expectation. |
| **3.5** | Sum | $E\!\left[\sum_{i=1}^{k} g_i(Y)\right] = \sum_{i=1}^{k} E[g_i(Y)]$ | The expectation of a sum is the sum of expectations. |

### The single idea behind all three: **expectation is linear**

Putting 3.4 and 3.5 together, for constants $a, b$:

$$E[a\,g(Y) + b] = a\,E[g(Y)] + b.$$

**Why it works (one line each):**
- **3.3**: $\sum_y c\,p(y) = c\sum_y p(y) = c\cdot 1 = c$.
- **3.4**: $\sum_y c\,g(y)\,p(y) = c\sum_y g(y)\,p(y) = c\,E[g(Y)]$.
- **3.5**: $\sum_y \bigl[g_1(y)+\cdots+g_k(y)\bigr]p(y) = \sum_y g_1(y)p(y) + \cdots + \sum_y g_k(y)p(y)$.

All three are just the distributive law applied to a finite sum. Linearity is what makes expectation so pleasant — and it holds **even when the variables are not independent** (a fact you'll lean on heavily in later chapters).

> **Contrast:** linearity is special to the *linear* combination $aY + b$. For a nonlinear $g$, in general $E[g(Y)] \ne g(E[Y])$ — exactly what we saw with $g(y) = y^2$.


In [ ]:
# Numerically verify E[aY + b] = a E[Y] + b on the fair die, for a = 3, b = -5.
y_vals = [1, 2, 3, 4, 5, 6]
p_vals = [1/6] * 6
a, b = 3, -5

lhs = expectation(y_vals, p_vals, g=lambda y: a*y + b)   # E[aY + b] directly
rhs = a * expectation(y_vals, p_vals) + b                # a E[Y] + b

print(f'E[aY + b]      = {lhs:.4f}')
print(f'a*E[Y] + b     = {rhs:.4f}')
print(f'match          = {np.isclose(lhs, rhs)}')

# And E[c] = c
c = 7.0
print(f'\nE[c] with c={c}: {expectation(y_vals, p_vals, g=lambda y: c):.4f}')


---

## 4. Variance and standard deviation

The mean tells you *where* the distribution sits. The **variance** tells you *how far* $Y$ typically strays from that center.

### Definition 3.5

The **variance** of $Y$ with mean $\mu = E[Y]$ is the expected squared deviation from the mean:

$$V(Y) \;=\; \sigma^2 \;=\; E\bigl[(Y - \mu)^2\bigr] \;=\; \sum_y (y - \mu)^2\, p(y).$$

The **standard deviation** is $\sigma = \sqrt{V(Y)}$, back in the original units of $Y$.

### Why *squared* deviations?

- The raw average deviation $E[Y - \mu]$ is always **0** (positives and negatives cancel exactly — that's what "balance point" means). Useless as a spread measure.
- Squaring makes every deviation non-negative, so they can't cancel, and it **punishes large deviations more** than small ones. Taking the square root at the end ($\sigma$) restores the original units.

### Theorem 3.6 — the computational shortcut

$$\boxed{\,V(Y) = E[Y^2] - \mu^2\,}$$

**Proof (pure linearity):** expand the square and use Theorems 3.3–3.5.

$$
V(Y) = E\bigl[(Y-\mu)^2\bigr] = E\bigl[Y^2 - 2\mu Y + \mu^2\bigr]
= E[Y^2] - 2\mu\,E[Y] + \mu^2
= E[Y^2] - 2\mu^2 + \mu^2 = E[Y^2] - \mu^2.
$$

In practice this is almost always the easier road: compute $E[Y]$ and $E[Y^2]$ (two simple sums), then subtract. Note $E[Y^2] - \mu^2 \ge 0$ always — a handy sanity check. If you ever get a negative "variance," you made an arithmetic slip.


In [ ]:
# Variance three ways on the fair die: definition, Theorem 3.6, and simulation.
y_vals = [1, 2, 3, 4, 5, 6]
p_vals = [1/6] * 6

mu  = expectation(y_vals, p_vals)
var_def   = expectation(y_vals, p_vals, g=lambda y: (y - mu)**2)   # Definition 3.5
var_short = expectation(y_vals, p_vals, g=lambda y: y**2) - mu**2  # Theorem 3.6

print(f'mean mu            = {mu:.4f}')
print(f'V(Y) definition    = {var_def:.4f}')
print(f'V(Y) Theorem 3.6   = {var_short:.4f}')
print(f'sigma              = {np.sqrt(var_short):.4f}')

# The raw (unsquared) average deviation is zero — why we square:
print(f'\nE[Y - mu] (should be 0) = {expectation(y_vals, p_vals, g=lambda y: y - mu):.6f}')

rolls = rng.integers(1, 7, size=500_000)
print(f'\nSimulated V(Y) approx {rolls.var():.4f}')


---

## 5. Worked exercises

### Exercise A — Expected number of correct matches (callback to Section 3.2)

Recall the children's matching problem: three words assigned at random to three animal pictures, $Y$ = number of correct matches. From Section 3.2 the distribution is

| $y$ | $0$ | $1$ | $2$ | $3$ |
|---|---|---|---|---|
| $p(y)$ | $1/3$ | $1/2$ | $0$ | $1/6$ |

**Find $E[Y]$ and $V(Y)$.**

**Mean.**

$$E[Y] = 0\cdot\tfrac13 + 1\cdot\tfrac12 + 2\cdot 0 + 3\cdot\tfrac16 = \tfrac12 + \tfrac12 = 1.$$

There is a beautiful reason the answer is exactly **1**: write $Y = I_1 + I_2 + I_3$, where $I_j = 1$ if picture $j$ is matched correctly and $0$ otherwise. Each picture is matched with probability $1/n = 1/3$, so $E[I_j] = 1/3$. By linearity (Theorem 3.5),

$$E[Y] = E[I_1] + E[I_2] + E[I_3] = 3\cdot\tfrac13 = 1.$$

This "indicator + linearity" trick gives $E[Y] = 1$ correct match on average **for any $n$**, matched or not — a lovely fact, and it needs no derangement counting at all.

**Variance.**

$$E[Y^2] = 0^2\cdot\tfrac13 + 1^2\cdot\tfrac12 + 2^2\cdot 0 + 3^2\cdot\tfrac16 = \tfrac12 + \tfrac{9}{6} = \tfrac12 + \tfrac32 = 2.$$

$$V(Y) = E[Y^2] - \mu^2 = 2 - 1^2 = 1, \qquad \sigma = 1.$$


In [ ]:
# Exercise A: matching problem — exact E, V, plus indicator-trick and simulation checks.
y_vals = [0, 1, 2, 3]
p_vals = [1/3, 1/2, 0, 1/6]
mu, var = summarize(y_vals, p_vals, name='Y (matches)')
plot_pmf_mean(y_vals, p_vals, 'Exercise A — number of correct matches', xlabel='y = correct matches')

# Simulation: random assignment of 3 words to 3 pictures; count fixed points.
correct = np.array([1, 2, 3])
N = 300_000
matches = np.empty(N, dtype=int)
for i in range(N):
    guess = rng.permutation(correct)
    matches[i] = np.sum(guess == correct)

print(f'\nSimulated E[Y] approx {matches.mean():.4f}   (exact 1)')
print(f'Simulated V(Y) approx {matches.var():.4f}   (exact 1)')


---

### Exercise B — A fair price to play (expected gain)

A game costs \$$c$ to enter. You draw one ball from an urn containing balls labeled with dollar prizes: three balls pay \$1, two balls pay \$5, and one ball pays \$10 (six balls total). Let $W$ be your **net gain**, $W = (\text{prize}) - c$.

**(i) What is the expected prize $E[Y]$? (ii) What entry fee $c$ makes the game fair, i.e. $E[W] = 0$?**

**Prize distribution.**

| prize $y$ | \$1 | \$5 | \$10 |
|---|---|---|---|
| $p(y)$ | $3/6$ | $2/6$ | $1/6$ |

**(i) Expected prize.**

$$E[Y] = 1\cdot\tfrac36 + 5\cdot\tfrac26 + 10\cdot\tfrac16 = \tfrac{3 + 10 + 10}{6} = \tfrac{23}{6} \approx 3.833.$$

**(ii) Fair fee.** The net gain is $W = Y - c$, a linear function, so by linearity (Theorem 3.4/3.5),

$$E[W] = E[Y] - c = \tfrac{23}{6} - c.$$

Setting $E[W] = 0$ gives the **fair fee** $c = \tfrac{23}{6} \approx \$3.83$. Charge more and the game favors the house; charge less and it favors the player. This is the everyday meaning of "expected value": the break-even price.


In [ ]:
# Exercise B: expected prize and the fair entry fee.
prizes = [1, 5, 10]
p_vals = [3/6, 2/6, 1/6]

EY = expectation(prizes, p_vals)
c_fair = EY
print(f'E[prize]          = {EY:.4f}  = 23/6')
print(f'fair entry fee c  = {c_fair:.4f}  (makes E[net gain] = 0)')

# Sanity: simulate net gain at the fair fee and at an unfair fee.
N = 400_000
draws = rng.choice(prizes, size=N, p=p_vals)
for c in [c_fair, 5.0]:
    net = draws - c
    verdict = 'fair' if np.isclose(c, c_fair) else 'house edge'
    print(f'  fee c = {c:5.2f}:  simulated E[net gain] approx {net.mean():+.4f}   ({verdict})')


---

### Exercise C — Expected profit, a nonlinear $g(Y)$

A stand's daily demand $Y$ (in hundreds of units) has the distribution below. The day's **profit** (in dollars) is

$$g(Y) = 100 + 20Y - 5Y^2,$$

which rises with demand but is dragged down by a congestion/overtime cost that grows like $Y^2$.

| demand $y$ | $0$ | $1$ | $2$ | $3$ |
|---|---|---|---|---|
| $p(y)$ | $0.1$ | $0.4$ | $0.3$ | $0.2$ |

**Find the expected daily profit $E[g(Y)]$.**

**Method 1 — Theorem 3.2 directly.** Weight $g(y)$ by $p(y)$:

$$E[g(Y)] = \sum_y \bigl(100 + 20y - 5y^2\bigr)p(y).$$

**Method 2 — split with linearity first**, then plug in the two moments $E[Y]$ and $E[Y^2]$:

$$E[g(Y)] = 100 + 20\,E[Y] - 5\,E[Y^2].$$

With $E[Y] = 0(0.1)+1(0.4)+2(0.3)+3(0.2) = 1.6$ and $E[Y^2] = 0(0.1)+1(0.4)+4(0.3)+9(0.2) = 3.4$:

$$E[g(Y)] = 100 + 20(1.6) - 5(3.4) = 100 + 32 - 17 = 115.$$

Both methods must agree — Method 2 just reuses moments you'd compute anyway. Note again that $E[g(Y)] = 115 \ne g(E[Y]) = g(1.6) = 100 + 32 - 12.8 = 119.2$: plugging the mean into a nonlinear $g$ gives the **wrong** answer.


In [ ]:
# Exercise C: expected profit two ways, and the g(E[Y]) != E[g(Y)] warning.
y_vals = [0, 1, 2, 3]
p_vals = [0.1, 0.4, 0.3, 0.2]
g = lambda y: 100 + 20*y - 5*y**2

EY  = expectation(y_vals, p_vals)
EY2 = expectation(y_vals, p_vals, g=lambda y: y**2)

Eg_direct = expectation(y_vals, p_vals, g=g)     # Method 1: Theorem 3.2
Eg_linear = 100 + 20*EY - 5*EY2                   # Method 2: linearity + moments

print(f'E[Y]  = {EY:.2f}')
print(f'E[Y^2]= {EY2:.2f}')
print(f'\nE[g(Y)] via Theorem 3.2      = {Eg_direct:.2f}')
print(f'E[g(Y)] via linearity split  = {Eg_linear:.2f}')
print(f'g(E[Y]) (the WRONG shortcut) = {g(EY):.2f}')

# Simulation
N = 500_000
demand = rng.choice(y_vals, size=N, p=p_vals)
print(f'\nSimulated E[g(Y)] approx {g(demand).mean():.2f}')


---

## Recap — the toolkit of Section 3.3

| Concept | Formula | When you reach for it |
|---|---|---|
| **Mean** (Def 3.4) | $E[Y] = \sum_y y\,p(y)$ | "Where is the center / long-run average?" |
| **Function** (Thm 3.2) | $E[g(Y)] = \sum_y g(y)\,p(y)$ | Expectation of winnings, cost, $Y^2$, ... — skip finding the dist. of $g(Y)$. |
| **Linearity** (Thm 3.3–3.5) | $E[aY + b] = aE[Y] + b$ | Factor out constants; split sums; **holds even for dependent variables**. |
| **Variance** (Def 3.5) | $V(Y) = E[(Y-\mu)^2]$ | "How spread out is it?" (squared units) |
| **Var. shortcut** (Thm 3.6) | $V(Y) = E[Y^2] - \mu^2$ | The practical way to compute variance. |
| **Std. deviation** | $\sigma = \sqrt{V(Y)}$ | Spread in the *original* units of $Y$. |

### Three warnings worth tattooing on your arm

1. $E[Y]$ **need not be an attainable value** (fair die: $3.5$).
2. For nonlinear $g$, $\;E[g(Y)] \ne g(E[Y])$. In particular $E[Y^2] \ne (E[Y])^2$ — the gap between them *is* the variance.
3. Linearity is only for **linear** combinations $aY + b$; do not distribute expectation across products or squares.

### The pattern for every problem

1. Write the pmf (support + probabilities), from Section 3.2.
2. Compute $E[Y]$ and $E[Y^2]$ — two weighted sums.
3. Read off the mean, and get variance from $E[Y^2] - \mu^2$.
4. Apply $g$ or linearity as the question demands, and simulate to check.
